In [1]:
!pip install pyspark -q

In [2]:
!pip install boto3 -q
!pip install pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [4]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [5]:
import requests
import boto3
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ── Session ───────────────────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("MyEarthQuakeApp") \
    .getOrCreate()

print("Spark version:", spark.version)

# ── Fetch API ─────────────────────────────────────────────────────────────────
response = requests.get("https://api.data.gov.my/weather/warning/earthquake")

# ── Parse JSON → Spark DataFrame ─────────────────────────────────────────────
rdd = spark.sparkContext.parallelize([response.text])
df = spark.read.json(rdd)

df.printSchema()

# ── Transform ─────────────────────────────────────────────────────────────────
df = df.withColumn(
    "utcdatetime_ts",
    F.expr("try_to_timestamp(utcdatetime, \"yyyy-MM-dd'T'HH:mm:ss\")")
)

df.orderBy(F.desc("utcdatetime_ts")).show(10)

# ── Write Parquet to dedicated subfolder ─────────────────────────────────────
local_dir  = "/tmp/earthquake"                          # subfolder, not root /tmp
local_path = os.path.join(local_dir, "earthquake_data.parquet")

os.makedirs(local_dir, exist_ok=True)                   # create if not exists

df.coalesce(1).write \
    .mode("overwrite") \
    .parquet(local_path)

# ── Find the actual .parquet part file Spark wrote ───────────────────────────
part_file = [
    f for f in os.listdir(local_path)
    if f.endswith(".parquet")
][0]

full_local_path = os.path.join(local_path, part_file)

# ── Upload to MinIO via boto3 ─────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

try:
    s3.upload_file(
        Filename=full_local_path,
        Bucket="rawdatasets",
        Key="earthquake/earthquake_data.parquet"
    )
    print("Uploaded to MinIO successfully!")
finally:
    # ── Cleanup: only delete /tmp/earthquake subfolder ────────────────────────
    shutil.rmtree(local_dir, ignore_errors=True)
    print(f"Deleted temp directory: {local_dir}")


Spark version: 4.0.2
root
 |-- depth: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- lat_vector: string (nullable = true)
 |-- localdatetime: string (nullable = true)
 |-- location: string (nullable = true)
 |-- location_original: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lon_vector: string (nullable = true)
 |-- magdefault: double (nullable = true)
 |-- magtypedefault: string (nullable = true)
 |-- n_distancemas: string (nullable = true)
 |-- n_distancerest: string (nullable = true)
 |-- nbm_distancemas: string (nullable = true)
 |-- nbm_distancerest: string (nullable = true)
 |-- status: string (nullable = true)
 |-- utcdatetime: string (nullable = true)
 |-- visible: boolean (nullable = true)

+-----+---------+----------+-------------------+--------------------+--------------------+-----------+-----------+----------+--------------+--------------------+--------------------+--------------------+--------------------+------+-------------------+--